In [2]:
# ============================================================
# CELL 0 — ENVIRONMENT SETUP (run this first, do not modify)
# ============================================================
# Target: Google Colab | GPU: T4 | RAM: 12GB+ | VRAM: 15GB
# Estimated total notebook runtime: 45–60 minutes
# ============================================================
import os
import sys

# --- Step 0: Force HuggingFace cache to /tmp (NOT Drive) ---
# This prevents caching ~1.8GB of model files into your 15GB Drive quota.
os.environ["TRANSFORMERS_CACHE"]   = "/tmp/hf_cache"
os.environ["HF_HOME"]              = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"]    = "/tmp/hf_datasets_cache"

# --- Step 1: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = "/content/drive/MyDrive/HateSpeech_NLP"

# --- Step 2: Clone or pull the Git repository ---
REPO_URL  = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    print("Cloning repository for the first time...")
    ret = os.system(f"git clone {REPO_URL} {REPO_PATH}")
    assert ret == 0, "❌ git clone failed! Check your REPO_URL."
else:
    print("Repository already exists. Pulling latest changes...")
    os.system(f"cd {REPO_PATH} && git pull origin main")

# --- Step 3: Install the project package in editable mode ---
ret = os.system(f"pip install -q -e {REPO_PATH}")
assert ret == 0, "❌ pip install -e failed!"

# --- Step 4: Add src/ to Python path ---
SRC_PATH = f"{REPO_PATH}/src"
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# --- Step 5: Install remaining dependencies ---
os.system(f"pip install -q -r {REPO_PATH}/requirements.txt")

# --- Final verification ---
print("=" * 55)
print("✅ Environment setup complete.")
print(f"   REPO_PATH  : {REPO_PATH}")
print(f"   DRIVE_ROOT : {DRIVE_ROOT}")
print(f"   SRC_PATH   : {SRC_PATH}")
print(f"   Python     : {sys.version.split()[0]}")
print("=" * 55)

# Sanity check: verify Drive is accessible
assert os.path.isdir(DRIVE_ROOT), (
    f"❌ Drive root not found: {DRIVE_ROOT}\n"
    "   Check that your Drive folder is 'HateSpeech_NLP'."
)
print("✅ Drive root verified.")


Mounted at /content/drive
Cloning repository for the first time...
✅ Environment setup complete.
   REPO_PATH  : /content/hate-speech-detection
   DRIVE_ROOT : /content/drive/MyDrive/HateSpeech_NLP
   SRC_PATH   : /content/hate-speech-detection/src
   Python     : 3.12.13
✅ Drive root verified.


In [ ]:
# ============================================================
# CELL 1 — GPU VERIFICATION & MEMORY UTILITIES
# ============================================================
# Input:  None
# Output: Confirms T4 GPU is active, defines memory helper functions
# ============================================================
import torch
import gc


def print_gpu_memory(tag: str = ""):
    """
    Print current GPU VRAM usage at a named checkpoint.
    Call this before and after every major operation.
    """
    if not torch.cuda.is_available():
        print(f"[GPU {tag}] ⚠️  No CUDA GPU detected.")
        return
    allocated = torch.cuda.memory_allocated() / 1024 ** 3
    reserved  = torch.cuda.memory_reserved()  / 1024 ** 3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    free      = total - reserved
    print(
        f"[GPU | {tag}] "
        f"Allocated: {allocated:.2f}GB | "
        f"Reserved: {reserved:.2f}GB | "
        f"Free: {free:.2f}GB / {total:.2f}GB"
    )


def clean_memory():
    """
    Force Python garbage collection and clear GPU cache.
    Call this before any large allocation (model load, dataset build, training).
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Memory cleaned (gc.collect + cuda.empty_cache).")


# --- GPU assertion ---
assert torch.cuda.is_available(), (
    "❌ CUDA GPU not found!\n"
    "   Fix: Runtime → Change runtime type → T4 GPU → Save → Reconnect."
)

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU detected : {gpu_name}")
print(f"   torch version: {torch.__version__}")
print_gpu_memory("initial")


In [ ]:
# ============================================================
# CELL 2 — LOAD CONFIG AND PROCESSED DATA
# ============================================================
# Input:
#   - {REPO_PATH}/configs/paths.yaml
#   - {DRIVE_ROOT}/data/processed/train.parquet
#   - {DRIVE_ROOT}/data/processed/dev.parquet      ← val set
#   - {DRIVE_ROOT}/data/processed/test.parquet
#   - {DRIVE_ROOT}/results/class_weights.json
# Output:
#   - train_df, val_df, test_df  (pandas DataFrames)
#   - class_weights              (list[float], len=3, ordered [CL, OFF, HATE])
#   - cfg                        (OmegaConf DictConfig)
# ============================================================
import json
import pandas as pd
from omegaconf import OmegaConf


def resolve_path(template_path) -> str:
    """
    Resolve the {drive_root} placeholder used in paths.yaml.
    Also handles the edge case where OmegaConf returns a nested dict
    instead of a plain string (robustness lesson from Phase 4).
    """
    if isinstance(template_path, str):
        return template_path.replace("{drive_root}", DRIVE_ROOT)
    # Unwrap DictConfig or dict values
    if hasattr(template_path, "values"):
        for v in template_path.values():
            return resolve_path(v)
    return str(template_path)


# --- Load config ---
cfg = OmegaConf.load(f"{REPO_PATH}/configs/paths.yaml")
print("✅ Config loaded from paths.yaml")

# --- Load parquet files ---
train_path = resolve_path(cfg.data.train_processed)
val_path   = resolve_path(cfg.data.val_processed)
test_path  = resolve_path(cfg.data.test_processed)

print(f"   Loading train : {train_path}")
print(f"   Loading val   : {val_path}")
print(f"   Loading test  : {test_path}")

train_df = pd.read_parquet(train_path)
val_df   = pd.read_parquet(val_path)
test_df  = pd.read_parquet(test_path)

# --- Schema validation ---
REQUIRED_COLUMNS = {"text", "label"}
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = REQUIRED_COLUMNS - set(df.columns)
    assert not missing, (
        f"❌ '{name}' parquet is missing columns: {missing}\n"
        f"   Found columns: {list(df.columns)}"
    )

print(f"\n✅ Data loaded successfully:")
print(f"   Train : {len(train_df):,} rows | Label dist: {train_df['label'].value_counts().to_dict()}")
print(f"   Val   : {len(val_df):,} rows   | Label dist: {val_df['label'].value_counts().to_dict()}")
print(f"   Test  : {len(test_df):,} rows  | Label dist: {test_df['label'].value_counts().to_dict()}")

# --- Load class weights ---
weights_path = resolve_path(cfg.results.class_weights)
with open(weights_path, "r") as f:
    class_weights_raw = json.load(f)

# Normalize to list format [w0, w1, w2] regardless of JSON structure
if isinstance(class_weights_raw, list):
    class_weights = [float(w) for w in class_weights_raw]
elif isinstance(class_weights_raw, dict):
    # Keys may be "0","1","2" or 0,1,2
    class_weights = [float(class_weights_raw[str(i)]) for i in range(3)]
else:
    raise ValueError(f"Unexpected class_weights format: {type(class_weights_raw)}")

assert len(class_weights) == 3, f"❌ Expected 3 class weights, got {len(class_weights)}"
print(f"\n✅ Class weights: [CLEAN={class_weights[0]:.4f}, OFFENSIVE={class_weights[1]:.4f}, HATE={class_weights[2]:.4f}]")


In [ ]:
# ============================================================
# CELL 3 — TOKENIZER & DATASET CREATION
# ============================================================
# Input:  train_df, val_df, test_df from Cell 2
# Output: tokenizer (XLM-RoBERTa),
#         train_dataset, val_dataset, test_dataset (ViHSDDataset)
# Note:   Tokenizer is downloaded to /tmp/hf_cache (NOT Drive)
# ============================================================
from transformers import AutoTokenizer
from data.dataset import ViHSDDataset   # src/data/dataset.py

MODEL_NAME = str(cfg.model.backbone)    # "xlm-roberta-base"
MAX_LENGTH = int(cfg.training.max_length)  # 128

clean_memory()
print(f"Loading tokenizer: '{MODEL_NAME}' → /tmp/hf_cache\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Build datasets ---
# Uses ViHSDDataset from src/data/dataset.py
# Each __getitem__ returns: input_ids, attention_mask, labels (torch tensors)
train_dataset = ViHSDDataset(
    texts=train_df["text"].tolist(),
    labels=train_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)
val_dataset = ViHSDDataset(
    texts=val_df["text"].tolist(),
    labels=val_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)
test_dataset = ViHSDDataset(
    texts=test_df["text"].tolist(),
    labels=test_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

# --- Sanity check: inspect one sample ---
sample = train_dataset[0]
assert set(sample.keys()) == {"input_ids", "attention_mask", "labels"}, (
    f"❌ Unexpected sample keys: {set(sample.keys())}"
)
assert sample["input_ids"].shape == (MAX_LENGTH,), (
    f"❌ input_ids shape mismatch: {sample['input_ids'].shape}"
)

print(f"✅ Tokenizer loaded: '{MODEL_NAME}'")
print(f"\n✅ Datasets created:")
print(f"   train_dataset : {len(train_dataset):,} samples")
print(f"   val_dataset   : {len(val_dataset):,} samples")
print(f"   test_dataset  : {len(test_dataset):,} samples")
print(f"\n   Sample[0] keys  : {list(sample.keys())}")
print(f"   input_ids shape : {sample['input_ids'].shape} (max_length={MAX_LENGTH})")
print(f"   label           : {sample['labels'].item()}")

print_gpu_memory("after_tokenizer_load")


In [ ]:
# ============================================================
# CELL 4 — MODEL LOADING + WEIGHTED TRAINER DEFINITION
# ============================================================
# Input:  MODEL_NAME (str), class_weights (list[float])
# Output: model on GPU with gradient_checkpointing enabled,
#         WeightedTrainer class (applies per-class loss weighting)
# VRAM after this cell: ~1.3 GB reserved (model is ~1.1 GB fp32)
# ============================================================
import torch
from transformers import AutoModelForSequenceClassification, Trainer

clean_memory()
print(f"Loading model: '{MODEL_NAME}' → /tmp/hf_cache ...")
print("(First run: ~1.1 GB download. Subsequent runs: instant from cache.)\n")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=cfg.model.num_labels,   # 3
    ignore_mismatched_sizes=True,      # Safe for classification head init
)

# --- CRITICAL: Enable gradient checkpointing ---
# Trades compute for memory: recomputes activations during backward pass.
# Reduces VRAM by ~50% → allows batch_size=16 on T4 without OOM.
model.gradient_checkpointing_enable()
model.config.use_cache = False   # Required when gradient_checkpointing=True

device = torch.device("cuda")
model = model.to(device)

param_count_M = model.num_parameters() / 1e6
print(f"✅ Model loaded on {device}")
print(f"   Parameters        : {param_count_M:.1f}M")
print(f"   Gradient checkpointing : ENABLED")
print(f"   use_cache         : {model.config.use_cache}  (must be False)")
print_gpu_memory("after_model_load")

# ===========================================================
# Custom Trainer that applies class-weighted CrossEntropyLoss.
# This directly combats the severe label imbalance in ViHSD
# (CLEAN >> HATE > OFFENSIVE).
# ===========================================================
class WeightedTrainer(Trainer):
    """
    HuggingFace Trainer subclass with class-weighted loss.

    The standard Trainer uses unweighted CrossEntropyLoss, which will
    make the model biased toward predicting CLEAN (the majority class).
    This subclass corrects that by weighting rare classes higher.
    """

    def __init__(self, class_weights: list, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Register as a plain tensor; move to correct device in compute_loss
        self._class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Move weights to the same device as logits (GPU)
        weights = self._class_weights.to(logits.device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


print(f"\n✅ WeightedTrainer class defined.")
print(f"   Class weights applied: {[round(w, 4) for w in class_weights]}")
print(f"   Mapping: [CLEAN, OFFENSIVE, HATE]")


In [ ]:
# ============================================================
# CELL 5 — TRAINING ARGUMENTS & CHECKPOINT RESUME DETECTION
# ============================================================
# Input:  cfg (OmegaConf), DRIVE_ROOT
# Output: training_args (TrainingArguments ready to pass to Trainer)
#         resume_checkpoint (str path or None)
#         CHECKPOINTS_DIR (Drive path where checkpoints are saved)
# ============================================================
import glob
import os
from transformers import TrainingArguments, EarlyStoppingCallback

# --- Checkpoint directory (on Drive, persists across sessions) ---
CHECKPOINTS_DIR = resolve_path(cfg.models.checkpoints_dir)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
print(f"Checkpoints directory: {CHECKPOINTS_DIR}")

# ===========================================================
# AUTO-RESUME FUNCTION
# Finds the most recent VALID checkpoint to resume from.
# A "valid" checkpoint must have trainer_state.json (written
# only after a COMPLETE epoch save, not a mid-epoch crash).
# ===========================================================
def find_latest_checkpoint(checkpoints_dir: str):
    """
    Scan the checkpoint directory and return the path to the most
    recent valid checkpoint, or None if none exist.

    A valid checkpoint directory contains:
      - trainer_state.json  (proof that the epoch completed)
      - pytorch_model.bin OR model.safetensors (actual weights)

    Args:
        checkpoints_dir: Path to the directory containing checkpoint-N/ folders.

    Returns:
        str path to the latest valid checkpoint, or None.
    """
    # Find all checkpoint-* subdirectories
    pattern = os.path.join(checkpoints_dir, "checkpoint-*")
    candidates = sorted(
        glob.glob(pattern),
        key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else 0,
    )

    if not candidates:
        print("   No checkpoint folders found.")
        return None

    print(f"   Found {len(candidates)} checkpoint folder(s):")
    for c in candidates:
        print(f"     {os.path.basename(c)}")

    # Walk backwards to find the most recent VALID checkpoint
    for checkpoint_path in reversed(candidates):
        has_state = os.path.exists(
            os.path.join(checkpoint_path, "trainer_state.json")
        )
        has_weights = (
            os.path.exists(os.path.join(checkpoint_path, "pytorch_model.bin"))
            or os.path.exists(os.path.join(checkpoint_path, "model.safetensors"))
        )

        if has_state and has_weights:
            return checkpoint_path
        else:
            missing = []
            if not has_state:
                missing.append("trainer_state.json")
            if not has_weights:
                missing.append("weights (pytorch_model.bin / model.safetensors)")
            print(f"   ⚠️  Skipping {os.path.basename(checkpoint_path)} — missing: {missing}")

    print("   No VALID checkpoint found. Will train from scratch.")
    return None


# --- Detect checkpoint ---
print("\n🔍 Scanning for resume checkpoint...")
resume_checkpoint = find_latest_checkpoint(CHECKPOINTS_DIR)

if resume_checkpoint:
    print(f"\n🔄 RESUME MODE: Will resume from → {os.path.basename(resume_checkpoint)}")
else:
    print(f"\n🆕 FRESH START: Will train from scratch (no valid checkpoint found).")

# ===========================================================
# TRAINING ARGUMENTS
# All values sourced from paths.yaml — no hardcoding here.
# Key Colab-safety settings highlighted with comments.
# ===========================================================
training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,

    # --- Training schedule ---
    num_train_epochs=cfg.training.num_epochs,                       # 10
    per_device_train_batch_size=cfg.training.batch_size,            # 16
    per_device_eval_batch_size=cfg.training.eval_batch_size,        # 32

    # --- Optimizer ---
    learning_rate=cfg.training.learning_rate,                       # 2e-5
    warmup_ratio=cfg.training.warmup_ratio,                         # 0.1
    weight_decay=cfg.training.weight_decay,                         # 0.01

    # --- Mixed precision (FP16): REQUIRED to fit on T4 ---
    fp16=bool(cfg.training.fp16),                                   # True

    # --- Evaluation & checkpointing ---
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=int(cfg.training.save_total_limit),            # 2
    load_best_model_at_end=bool(cfg.training.load_best_model_at_end),  # True
    metric_for_best_model=str(cfg.training.metric_for_best_model),     # "eval_f1_macro"
    greater_is_better=True,

    # --- Logging ---
    logging_dir=f"{CHECKPOINTS_DIR}/logs",
    logging_steps=50,                  # Print training loss every 50 steps
    report_to="none",                  # Disable W&B / TensorBoard (conflicts on Colab)

    # --- CRITICAL Colab safety settings ---
    dataloader_num_workers=int(cfg.training.dataloader_num_workers),  # 0 (MUST be 0)
    dataloader_pin_memory=False,        # Disable pinning (unnecessary + uses RAM)

    # --- Gradient checkpointing compatibility ---
    # This suppresses the DDP warning about unused parameters
    # when gradient_checkpointing is enabled.
    ddp_find_unused_parameters=False,

    # --- Disable safetensors for compatibility with older trainer versions ---
    # Comment this out if you see "safetensors not installed" warnings
    # save_safetensors=False,
)

# --- Summary printout ---
print("\n" + "=" * 55)
print("✅ TrainingArguments configured:")
print(f"   Output dir          : {training_args.output_dir}")
print(f"   Num epochs          : {training_args.num_train_epochs}")
print(f"   Train batch size    : {training_args.per_device_train_batch_size}")
print(f"   Eval batch size     : {training_args.per_device_eval_batch_size}")
print(f"   Learning rate       : {training_args.learning_rate}")
print(f"   FP16                : {training_args.fp16}  ← MUST be True")
print(f"   num_workers         : {training_args.dataloader_num_workers}  ← MUST be 0")
print(f"   eval_strategy       : {training_args.eval_strategy}")
print(f"   save_total_limit    : {training_args.save_total_limit}")
print(f"   metric_for_best     : {training_args.metric_for_best_model}")
print(f"   Resume checkpoint   : {os.path.basename(resume_checkpoint) if resume_checkpoint else 'None (fresh start)'}")
print("=" * 55)
print_gpu_memory("cell5_end")


In [ ]:
# ============================================================
# CELL 6 — TRAINING LOOP
# ============================================================
# ⏱️  ESTIMATED TIME : 45–60 minutes on T4 GPU (10 epochs)
# 💾  AUTO-SAVE      : Checkpoints saved to Drive after every epoch
# 🔄  RESUME         : If Colab crashes → re-run Cell 0→6
#                      (Cell 5 auto-detects the latest checkpoint)
#
# Input:
#   - model           : XLM-RoBERTa (278M params, on GPU) from Cell 4
#   - training_args   : TrainingArguments from Cell 5
#   - train_dataset   : ViHSDDataset (train) from Cell 3
#   - val_dataset     : ViHSDDataset (val)   from Cell 3
#   - class_weights   : list[float] from Cell 2
#   - resume_checkpoint: str or None from Cell 5
#
# Output:
#   - trainer         : Trained WeightedTrainer (best model loaded)
#   - train_result    : TrainOutput object with loss & step stats
# ============================================================
from transformers import EarlyStoppingCallback
from models.classifier import compute_metrics  # src/models/classifier.py

# EarlyStoppingCallback: stop if val F1 does not improve for 3 consecutive epochs.
# This prevents overfitting and saves training time.
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.001,   # Minimum improvement to count as "better"
)

# Instantiate WeightedTrainer (defined in Cell 4)
trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

# Pre-training cleanup: free every byte of VRAM before the long run
clean_memory()
print_gpu_memory("pre_training")

print()
print("=" * 60)
print("🚀 STARTING TRAINING")
print()
if resume_checkpoint:
    print(f"   Mode     : RESUME from {os.path.basename(resume_checkpoint)}")
else:
    print(f"   Mode     : FRESH START (epoch 1 of {training_args.num_train_epochs})")
print(f"   Epochs   : {training_args.num_train_epochs} (early stop patience=3)")
print(f"   Batch    : {training_args.per_device_train_batch_size}")
print(f"   FP16     : {training_args.fp16}")
print(f"   Checkpts : {CHECKPOINTS_DIR}")
print()
print("   ⚠️  DO NOT interrupt this cell unless Colab disconnects.")
print("   ⚠️  If disconnected: re-run Cells 0→6, it will auto-resume.")
print("=" * 60)

# --- THE TRAINING CALL ---
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

# --- Post-training summary ---
print()
print("=" * 60)
print("✅ TRAINING COMPLETE")
print(f"   Global steps   : {train_result.global_step:,}")
print(f"   Training loss  : {train_result.training_loss:.4f}")
print(f"   Epochs trained : {train_result.global_step / (len(train_dataset) / training_args.per_device_train_batch_size):.1f}")
print("=" * 60)
print_gpu_memory("post_training")

# Print full training history (loss + val metrics per epoch)
print()
print("📊 Training History (per epoch):")
print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val F1 (macro)':>14} | {'Val Loss':>10}")
print("-" * 55)

log_history = trainer.state.log_history
# Separate train_loss rows and eval rows
train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs  = [x for x in log_history if "eval_f1_macro" in x]

for i, eval_log in enumerate(eval_logs):
    epoch      = eval_log.get("epoch", i + 1)
    val_f1     = eval_log.get("eval_f1_macro", "—")
    val_loss   = eval_log.get("eval_loss", "—")
    # Match training loss from the same epoch interval
    matching_train = [
        t["loss"] for t in train_logs
        if abs(t.get("epoch", 0) - epoch) < 1.05
    ]
    t_loss = round(matching_train[-1], 4) if matching_train else "—"
    print(f"{epoch:>6.1f} | {str(t_loss):>12} | {str(val_f1):>14} | {str(round(val_loss, 4) if isinstance(val_loss, float) else val_loss):>10}")

# Best model info
best_ckpt = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric
print()
print(f"🏆 Best checkpoint : {os.path.basename(best_ckpt) if best_ckpt else 'last'}")
print(f"🏆 Best val F1     : {best_metric:.4f}" if isinstance(best_metric, float) else f"🏆 Best val F1: {best_metric}")


In [ ]:
# ============================================================
# CELL 7 — SAVE FINAL MODEL & (OPTIONAL) PUSH TO HF HUB
# ============================================================
# Input:
#   - trainer   : Trained WeightedTrainer (best model already loaded)
#   - tokenizer : XLM-RoBERTa tokenizer from Cell 3
# Output:
#   - {DRIVE_ROOT}/models/xlmr_vihsd/final_hf/  (HuggingFace format)
#   - (Optional) Model pushed to HuggingFace Hub
#   - Old checkpoint folders deleted to free Drive space
# ============================================================
import shutil
import glob
import json

FINAL_HF_DIR = resolve_path(cfg.models.final_hf_dir)
os.makedirs(FINAL_HF_DIR, exist_ok=True)

# --- Step 1: Save model + tokenizer to Drive in HuggingFace format ---
print(f"💾 Saving final model to Drive...")
print(f"   Path: {FINAL_HF_DIR}")

trainer.save_model(FINAL_HF_DIR)
tokenizer.save_pretrained(FINAL_HF_DIR)

# Verify the save
expected_files = ["config.json", "tokenizer.json", "tokenizer_config.json"]
for fname in expected_files:
    fpath = os.path.join(FINAL_HF_DIR, fname)
    if os.path.exists(fpath):
        print(f"   ✅ {fname}")
    else:
        print(f"   ⚠️  {fname} not found — may use safetensors format, check manually")

# Also save the best model metric for traceability
meta = {
    "backbone": str(cfg.model.backbone),
    "best_val_f1_macro": trainer.state.best_metric,
    "best_checkpoint": os.path.basename(trainer.state.best_model_checkpoint or ""),
    "class_weights": class_weights,
    "max_length": int(cfg.training.max_length),
    "num_epochs_trained": len([x for x in trainer.state.log_history if "eval_f1_macro" in x]),
}
meta_path = os.path.join(FINAL_HF_DIR, "training_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"   ✅ training_meta.json (best F1={meta['best_val_f1_macro']})")
print(f"\n✅ Model saved to Drive: {FINAL_HF_DIR}")

# --- Step 2: (Optional) Push to HuggingFace Hub ---
# Fill in HF_MODEL_ID below, then uncomment the push block.
# Format: "<your-hf-username>/vihsd-xlmr-hate-speech"
#
# HOW TO GET HF TOKEN:
#   1. Go to https://huggingface.co/settings/tokens
#   2. Create a token with WRITE permission
#   3. In Colab: Tools → Secrets → Add "HF_TOKEN"

HF_MODEL_ID = "thong7d/vihsd-xlmr-hate-speech"  # ← Fill this in! E.g. "thong7d/vihsd-xlmr-hate-speech"

if HF_MODEL_ID:
    print(f"\n📤 Pushing to HuggingFace Hub: {HF_MODEL_ID}")
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
        print("   ⚠️  HF_TOKEN not found in Colab Secrets. Attempting anonymous push...")

    try:
        trainer.model.push_to_hub(HF_MODEL_ID, token=hf_token)
        tokenizer.push_to_hub(HF_MODEL_ID, token=hf_token)
        print(f"   ✅ Pushed successfully!")
        print(f"   🌐 View at: https://huggingface.co/{HF_MODEL_ID}")

        # Update paths.yaml in memory so downstream cells can use it
        # ACTION REQUIRED: Manually update hf_model_id in configs/paths.yaml on your PC,
        # then commit and push to Git.
        print(f"\n   ⚠️  ACTION REQUIRED: Update configs/paths.yaml on your PC:")
        print(f'   hf_model_id: "{HF_MODEL_ID}"')
        print(f"   Then: git add configs/paths.yaml && git commit -m 'feat: add hf_model_id' && git push")

    except Exception as e:
        print(f"   ❌ Push failed: {e}")
        print(f"   Model is safely saved on Drive. Push manually later:")
        print(f"   from transformers import AutoModelForSequenceClassification, AutoTokenizer")
        print(f"   model = AutoModelForSequenceClassification.from_pretrained('{FINAL_HF_DIR}')")
        print(f"   model.push_to_hub('{HF_MODEL_ID}', token='<your_token>')")
else:
    print("\n   ℹ️  HF_MODEL_ID is empty → skipping Hub push.")
    print("   To push later: fill in HF_MODEL_ID and re-run this cell.")

# --- Step 3: Delete old checkpoint folders to free Drive space ---
# (Only keep final_hf, which is the exported model)
print(f"\n🗑️  Cleaning up intermediate checkpoints from Drive...")
checkpoint_pattern = os.path.join(CHECKPOINTS_DIR, "checkpoint-*")
deleted = 0
for ckpt_dir in glob.glob(checkpoint_pattern):
    shutil.rmtree(ckpt_dir, ignore_errors=True)
    print(f"   Deleted: {os.path.basename(ckpt_dir)}")
    deleted += 1

if deleted == 0:
    print("   No checkpoint folders to delete.")
else:
    print(f"   ✅ {deleted} checkpoint folder(s) removed → Drive space freed.")

clean_memory()
print_gpu_memory("after_save")


In [ ]:
# ============================================================
# CELL 8 — TEST SET EVALUATION & SAVE REPORT
# ============================================================
# Input:
#   - trainer      : Trained WeightedTrainer (best model loaded)
#   - test_dataset : ViHSDDataset (test split) from Cell 3
#   - cfg          : OmegaConf config
# Output:
#   - Test metrics printed (Macro F1, per-class F1)
#   - {DRIVE_ROOT}/results/finetune_report.json saved to Drive
#
# Note: This is a quick validation only.
# Full evaluation (confusion matrix, error analysis) is in 06_evaluation.ipynb
# ============================================================
import json
import os

# --- Step 1: Run evaluation on test set ---
print("🔍 Evaluating on test set...")
test_results = trainer.evaluate(test_dataset)

# --- Step 2: Load baseline metrics for comparison ---
baseline_report_path = resolve_path(cfg.results.baseline_report)
baseline_f1 = "N/A"
if os.path.exists(baseline_report_path):
    try:
        with open(baseline_report_path, "r") as f:
            baseline_data = json.load(f)
        # Handle various possible keys from Phase 4 notebook
        baseline_f1 = (
            baseline_data.get("macro_f1")
            or baseline_data.get("f1_macro")
            or baseline_data.get("test_macro_f1")
            or "N/A"
        )
    except Exception as e:
        print(f"   ⚠️  Could not read baseline report: {e}")
else:
    print(f"   ⚠️  Baseline report not found at: {baseline_report_path}")

# --- Step 3: Display comparison ---
ft_f1      = test_results.get("eval_f1_macro", "N/A")
ft_clean   = test_results.get("eval_f1_clean", "N/A")
ft_off     = test_results.get("eval_f1_offensive", "N/A")
ft_hate    = test_results.get("eval_f1_hate", "N/A")
ft_loss    = test_results.get("eval_loss", "N/A")

print()
print("=" * 60)
print("📊 PHASE 5 RESULTS — XLM-RoBERTa Fine-tuned")
print("=" * 60)
print(f"   {'Metric':<22} {'Baseline':>12} {'Fine-tuned':>12}")
print(f"   {'-'*46}")
print(f"   {'Macro F1':<22} {str(baseline_f1):>12} {str(ft_f1):>12}")
print(f"   {'F1 — CLEAN':<22} {'N/A':>12} {str(ft_clean):>12}")
print(f"   {'F1 — OFFENSIVE':<22} {'N/A':>12} {str(ft_off):>12}")
print(f"   {'F1 — HATE':<22} {'N/A':>12} {str(ft_hate):>12}")
print(f"   {'Eval Loss':<22} {'N/A':>12} {str(round(ft_loss, 4) if isinstance(ft_loss, float) else ft_loss):>12}")
print("=" * 60)

# Improvement check
try:
    improvement = float(ft_f1) - float(baseline_f1)
    sign = "+" if improvement >= 0 else ""
    print(f"\n   📈 Improvement over baseline: {sign}{improvement:.4f} Macro F1")
    if improvement >= 0.10:
        print("   🎉 Excellent! >10% improvement achieved.")
    elif improvement >= 0.05:
        print("   ✅ Good improvement (5–10%).")
    else:
        print("   ⚠️  Small improvement (<5%). Consider checking class weights or learning rate.")
except (ValueError, TypeError):
    pass

# --- Step 4: Save report to Drive (atomic write via temp file) ---
finetune_report = {
    "phase": "phase_5_finetune",
    "model_backbone": str(cfg.model.backbone),
    "hf_model_id": str(cfg.get("hf_model_id", "")),
    "max_length": int(cfg.training.max_length),
    "num_epochs_trained": meta.get("num_epochs_trained", "N/A"),
    "best_val_f1_macro": meta.get("best_val_f1_macro"),
    "best_checkpoint": meta.get("best_checkpoint", ""),
    # Test set results
    "test_macro_f1": ft_f1,
    "test_f1_clean": ft_clean,
    "test_f1_offensive": ft_off,
    "test_f1_hate": ft_hate,
    "test_eval_loss": round(ft_loss, 4) if isinstance(ft_loss, float) else ft_loss,
    # Baseline for reference
    "baseline_macro_f1": baseline_f1,
    "class_weights": class_weights,
}

finetune_report_path = resolve_path(cfg.results.finetune_report)
os.makedirs(os.path.dirname(finetune_report_path), exist_ok=True)

# Atomic write: write to temp → move to target (safe even if Drive is slow)
import tempfile
tmp_fd, tmp_path = tempfile.mkstemp(
    dir=os.path.dirname(finetune_report_path), suffix=".json"
)
try:
    with os.fdopen(tmp_fd, "w") as f:
        json.dump(finetune_report, f, indent=2, ensure_ascii=False)
    shutil.move(tmp_path, finetune_report_path)
    print(f"\n✅ Report saved: {finetune_report_path}")
except Exception as e:
    if os.path.exists(tmp_path):
        os.remove(tmp_path)
    raise RuntimeError(f"Failed to save finetune report: {e}")

# --- Step 5: Final cleanup ---
del trainer, model
clean_memory()
print_gpu_memory("phase5_complete")

print()
print("=" * 60)
print("🎉 PHASE 5 COMPLETE!")
print()
print("   Next steps:")
print("   1. Note your test Macro F1 value above.")
print("   2. If HF_MODEL_ID is not yet filled, set it in Cell 7 and re-run.")
print("   3. Commit updated paths.yaml (with hf_model_id) to Git.")
print("   4. Proceed to: notebooks/06_evaluation.ipynb")
print("=" * 60)


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from google.colab import userdata

FINAL_HF_DIR = "/content/drive/MyDrive/HateSpeech_NLP/models/xlmr_vihsd/final_hf"
HF_MODEL_ID = "thong7d/vihsd-xlmr-hate-speech"

print("Đang tải model từ Google Drive...")
model = AutoModelForSequenceClassification.from_pretrained(FINAL_HF_DIR)
tokenizer = AutoTokenizer.from_pretrained(FINAL_HF_DIR)

print(f"Đang đẩy lên HuggingFace Hub: {HF_MODEL_ID} ...")
try:
    hf_token = userdata.get("HF_TOKEN")
    model.push_to_hub(HF_MODEL_ID, token=hf_token)
    tokenizer.push_to_hub(HF_MODEL_ID, token=hf_token)
    print(f"✅ Pushed successfully! Xem tại: https://huggingface.co/{HF_MODEL_ID}")
except Exception as e:
    print(f"❌ Lỗi: {e}. (Hãy chắc chắn bạn đã cấp quyền Write cho Token và thêm vào Secrets)")